# Env Setup

In [1]:
#!apt-get update -qq
#!apt-get install -y openjdk-8-jdk-headless -qq
#!wget -q https://archive.apache.org/dist/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz

#!tar -xzf spark-3.5.8-bin-hadoop3.tgz
#!pip install -q findspark

In [2]:
#!wget https://snap.stanford.edu/data/web-Google.txt.gz
#!gunzip -f ./web-Google.txt.gz

In [3]:
#!pip install graphframes-py==0.10.0

In [4]:
#!pip install sparkmeasure

# Dataset Initialization

In [1]:
import pyspark
import math
import builtins
from pyspark.sql import *
#from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf
from pyspark.sql import functions
from pyspark.sql.functions import lit, coalesce
import graphframes as gf
from sparkmeasure import StageMetrics
spark = SparkSession.builder \
    .config("spark.jars.packages",
            "io.graphframes:graphframes-spark4_2.13:0.10.0,"
            "ch.cern.sparkmeasure:spark-measure_2.13:0.24") \
    .getOrCreate()

sc = SparkContext.getOrCreate()
stagemetrics = StageMetrics(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/04 22:19:41 WARN Utils: Your hostname, Liams-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.148 instead (on interface en0)
26/04/04 22:19:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/liamshatzel/code/csc502/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/liamshatzel/.ivy2.5.2/cache
The jars for the packages stored in: /Users/liamshatzel/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
ch.cern.sparkmeasure#spark-measure_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bf2b9128-25bb-4c52-a4f7-ac9e9bac5695;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.10.0 in central
	found io.graphframes#graphframes-graphx-spark4_2.1

In [2]:
# Load and parse web_rdd dataset
web_rdd = sc.textFile("small-web-Google.txt")
header_lines = web_rdd.take(4)
edge_list = web_rdd.filter(lambda x: x not in header_lines).map(lambda x: x.split('\t')).map(lambda x: (int(x[0]), int(x[1])))

In [3]:
# Load and parse small dataset
edges = [
        (1, 2),
        (1, 3),
        (4, 3),
        (2, 3),
        (2, 7),
        (3, 7),
        (7, 8),
        (7, 6),
        (8, 5),
        (8, 2),
        (9, 7),
        (5, 9),
        (6, 5),
        (5, 6),
        (6, 9)
]

web_rdd2 = sc.parallelize(edges)
print(web_rdd2.collect())

[(1, 2), (1, 3), (4, 3), (2, 3), (2, 7), (3, 7), (7, 8), (7, 6), (8, 5), (8, 2), (9, 7), (5, 9), (6, 5), (5, 6), (6, 9)]


# PageRank

In [3]:
def preprocess_page_rank(web_rdd):
  edge_list = web_rdd

  outgoing_nodes = edge_list.groupByKey()
  out_degrees = outgoing_nodes.map(lambda x: (x[0], len(x[1])))

  incoming_nodes = edge_list.map(lambda x: (x[1], 1)).groupByKey()
  total_vertices = outgoing_nodes.union(incoming_nodes)

  # M(i, j, 1/N)
  M = edge_list.join(out_degrees).map(lambda x: (x[0], x[1][0], 1 / x[1][1]))

  vert_count = total_vertices.count()
  v = total_vertices.map(lambda x: (x[0], 1.0 / vert_count))

  return M, v

## PageRank without Caching RDDs simulating MapReduce

In [9]:
def page_rank(init_M, init_v, threshold, max_iteration=100):
  v_old = init_v
  v_new = init_v

  converged = False
  iteration_count = 0
  while not converged and iteration_count <= max_iteration:
    iteration_count += 1

    contributions = init_M.map(lambda x: (x[0], (x[1],x[2]))) \
                          .join(v_new) \
                          .map(lambda x: (x[1][0][0], x[1][0][1] * x[1][1]))

    v_new_no_sinks = contributions.groupByKey().mapValues(lambda x: builtins.sum(x))

    v_new = init_v.leftOuterJoin(v_new_no_sinks).mapValues(lambda x: (x[1] if x[1] is not None else 0.0))

    diff = v_new.join(v_old) \
                          .map(lambda x: builtins.abs(x[1][0] - x[1][1])) \
                          .sum()

    if diff < threshold:
      converged = True
    else:
      v_old = v_new

  return v_new, iteration_count

In [12]:
# Raw RDDs simulating MapReduce: no caching, re-read graph every iteration
M, v_dict = preprocess_page_rank(edge_list)

# Measure page rank
stagemetrics.begin()
pg_vals, number_of_iterations_pagerank = page_rank(M, v_dict, 0.001)
stagemetrics.end()
stagemetrics.print_report()

[Stage 8208:===================================================>(843 + 7) / 850]


Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 356
numTasks => 107214
elapsedTime => 2668709 (44 min)
stageDuration => 2659109 (44 min)
executorRunTime => 25880753 (7.2 h)
executorCpuTime => 313354 (5.2 min)
executorDeserializeTime => 65898 (1.1 min)
executorDeserializeCpuTime => 76803 (1.3 min)
resultSerializationTime => 41548 (42 s)
jvmGCTime => 400055 (6.7 min)
shuffleFetchWaitTime => 13 (13 ms)
shuffleWriteTime => 19761 (20 s)
resultSize => 2672969 (2.5 MB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 19936046400
recordsRead => 1330
bytesRead => 57224 (55.9 KB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 32668
shuffleTotalBlocksFetched => 32522
shuffleLocalBlocksFetched => 32522
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 2821225 (2.7 MB)
shuffleLocalBytesRead => 2821225 (2.7 MB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemot

## Uncached PageRank with Lin-Schatz Optimizations

In [8]:
def ls_page_rank(init_M, init_v, threshold, partitions=1, max_iteration=100):
    # Range partition
    # Lambda function key % partition does the range part here
    init_M = init_M.map(lambda x: (x[0], (x[1], x[2]))).partitionBy(partitions, lambda key: key % partitions)
    init_v = init_v.partitionBy(partitions, lambda key: key % partitions)
    
    v_old = init_v
    v_new = init_v

    converged = False
    iteration_count = 0
    while not converged and iteration_count <= max_iteration:
        iteration_count += 1
    
        contributions = init_M.join(v_new) \
            .map(lambda x: (x[1][0][0], x[1][0][1] * x[1][1]))
    
        v_new_no_sinks = contributions.reduceByKey(lambda a, b: a + b)
    
        v_new = init_v.leftOuterJoin(v_new_no_sinks).mapValues(lambda x: (x[1] if x[1] is not None else 0.0))
    
        diff = v_new.join(v_old) \
            .map(lambda x: builtins.abs(x[1][0] - x[1][1])) \
            .sum()
    
        if diff < threshold:
            converged = True
        else:
            v_old = v_new
    
    return v_new, iteration_count

In [9]:
# Raw RDDs simulating MapReduce: no caching, re-read graph every iteration
M, v_dict = preprocess_page_rank(edge_list)

# Measure page rank
stagemetrics.begin()
pg_vals, number_of_iterations_pagerank = ls_page_rank(M, v_dict, 0.001)
stagemetrics.end()
stagemetrics.print_report()

[Stage 3439:===============================================>   (168 + 10) / 180]


Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 228
numTasks => 14545
elapsedTime => 315876 (5.3 min)
stageDuration => 311595 (5.2 min)
executorRunTime => 2921748 (49 min)
executorCpuTime => 57278 (57 s)
executorDeserializeTime => 6800 (7 s)
executorDeserializeCpuTime => 9557 (10 s)
resultSerializationTime => 7539 (8 s)
jvmGCTime => 42510 (43 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 18919 (19 s)
resultSize => 439549 (429.2 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 70
bytesRead => 734 (734 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 19254
shuffleTotalBlocksFetched => 18962
shuffleLocalBlocksFetched => 18962
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 1652943 (1614.2 KB)
shuffleLocalBytesRead => 1652943 (1614.2 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 By

## PageRank with Cached RDD

In [39]:
def page_rank_caching(init_M, init_v, threshold, max_iteration=100):
    init_M.cache()
    init_M.count()

    # Cache graph structure
    adj_list = init_M.map(lambda x: (x[0], (x[1], x[2]))).groupByKey().mapValues(list)
    adj_list.cache()
    adj_list.count()
    
    init_v.cache()
    init_v.count()
        
    v_new = init_v
    converged = False
    iteration_count = 0
    while not converged and iteration_count <= max_iteration:
        iteration_count += 1
        v_old = v_new

        contributions = adj_list.join(v_old) \
                              .map(lambda x: (x[1][0][0], x[1][0][1] * x[1][1]))
        
        v_new_no_sinks = contributions.groupByKey().mapValues(lambda x: builtins.sum(x))

        v_new = init_v.leftOuterJoin(v_new_no_sinks) \
                      .mapValues(lambda x: (x[1] if x[1] is not None else 0.0))
        v_new.cache()
        v_new.count()
        
        diff = v_new.join(v_old) \
                    .map(lambda x: abs(x[1][0] - x[1][1])) \
                    .sum()

        if v_old is not init_v:
            v_old.unpersist()

        if diff < threshold:
            converged = True

    return v_new, iteration_count

[Stage 9231:==========================>                      (2246 + 10) / 4094]

In [14]:
M, v = preprocess_page_rank(edge_list)

# Measure page rank
stagemetrics.begin()
pg_vals_cached, number_of_iterations_pagerank_cached = page_rank_caching(M, v, 0.001)
stagemetrics.end()
stagemetrics.print_report()

[Stage 756:====================================================>(128 + 2) / 130]


Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 69
numTasks => 3178
elapsedTime => 74776 (1.2 min)
stageDuration => 73318 (1.2 min)
executorRunTime => 671862 (11 min)
executorCpuTime => 12472 (12 s)
executorDeserializeTime => 1748 (2 s)
executorDeserializeCpuTime => 2070 (2 s)
resultSerializationTime => 1422 (1 s)
jvmGCTime => 8810 (9 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 3437 (3 s)
resultSize => 277789 (271.3 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 1563
bytesRead => 90031 (87.9 KB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2974
shuffleTotalBlocksFetched => 2948
shuffleLocalBlocksFetched => 2948
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 253365 (247.4 KB)
shuffleLocalBytesRead => 253365 (247.4 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleByt

## Cached PageRank with Lin-Schatz Optimizations

In [26]:
def ls_page_rank_caching(init_M, init_v, threshold, partitions=1, max_iteration=100):
    
    # Range partition
    # Lambda function key % partition does the range part here
    init_M = init_M.map(lambda x: (x[0], (x[1], x[2]))).partitionBy(partitions, lambda key: key % partitions).cache()
    init_v = init_v.partitionBy(partitions, lambda key: key % partitions).cache()
    
    if not init_M.is_cached:
        init_M.cache()
        init_M.count()
    v_new = init_v
    converged = False
    iteration_count = 0
    while not converged and iteration_count <= max_iteration:
        iteration_count += 1
        v_old = v_new

        contributions = init_M.join(v_old) \
                              .map(lambda x: (x[1][0][0], x[1][0][1] * x[1][1]))
        
        # In-mapper combining (reduce network overhead)
        v_new_no_sinks = contributions.reduceByKey(lambda a, b: a + b)

        v_new = init_v.leftOuterJoin(v_new_no_sinks) \
                      .mapValues(lambda x: (x[1] if x[1] is not None else 0.0))
        v_new.cache()
        v_new.count()
        
        diff = v_new.join(v_old) \
                    .map(lambda x: abs(x[1][0] - x[1][1])) \
                    .sum()

        if v_old is not init_v:
            v_old.unpersist()

        if diff < threshold:
            converged = True

    return v_new, iteration_count

In [27]:
M, v = preprocess_page_rank(edge_list)

# Measure page rank
stagemetrics.begin()
pg_vals_cached, number_of_iterations_pagerank_cached = ls_page_rank_caching(M, v, 0.001, 1, 10)
stagemetrics.end()
stagemetrics.print_report()

[Stage 8794:=============================================>        (37 + 7) / 44]


Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 69
numTasks => 1088
elapsedTime => 32188 (32 s)
stageDuration => 31087 (31 s)
executorRunTime => 253047 (4.2 min)
executorCpuTime => 5354 (5 s)
executorDeserializeTime => 1463 (1 s)
executorDeserializeCpuTime => 711 (0.7 s)
resultSerializationTime => 704 (0.7 s)
jvmGCTime => 4704 (5 s)
shuffleFetchWaitTime => 2 (2 ms)
shuffleWriteTime => 1907 (2 s)
resultSize => 94824 (92.6 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 1180
bytesRead => 55257 (54.0 KB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2420
shuffleTotalBlocksFetched => 2402
shuffleLocalBlocksFetched => 2402
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 212808 (207.8 KB)
shuffleLocalBytesRead => 212808 (207.8 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesWritt

## PageRank with Graph Structure in GraphX

In [28]:
def process_with_graph(web_rdd, max_iter=10):
    # Pregel message passing with graph frames (pyspark version of GraphX)

    # create columns for vertices and edges
    total_vertices = web_rdd.count()

    outgoing_nodes = web_rdd.groupByKey().map(lambda x: (x[0],)).collect()

    vertices = spark.createDataFrame(outgoing_nodes, ["id"])

    edges = spark.createDataFrame(web_rdd.collect(), ["src", "dst"])

    # Build graph using graph frame
    g = gf.GraphFrame(vertices, edges)
    
    # create out degree per vertex
    out_degree = g.outDegrees
    
    # create new column out degree, tracking out degree per vertex
    out_vertices = g.vertices.join(out_degree, "id", "left_outer").fillna(0.0, subset=["outDegree"])
    
    # rebuild graph with out degrees
    g = gf.GraphFrame(out_vertices, edges)
    
    # define message passing
    init_g = g.pregel.setMaxIter(max_iter).withVertexColumn("pagerank", initialExpr=lit(1.0 / total_vertices), updateAfterAggMsgsExpr=coalesce(gf.lib.Pregel.msg(), lit(0.0)))
    
    # Message passsing step
    msg_g = init_g.sendMsgToDst(gf.lib.Pregel.src("pagerank") / gf.lib.Pregel.src("outDegree"))

    # TODO: Unoptimized version
    
    # Aggregate across messages
    agg_g = msg_g.aggMsgs(functions.sum(gf.lib.Pregel.msg()))
    
    # execute message passing
    ranked_verts = agg_g.run()
    
    return ranked_verts

In [ ]:
sc.setCheckpointDir("/tmp/graphframes-checkpoints")
stagemetrics.begin()
process_with_graph(edge_list)
stagemetrics.end()
stagemetrics.print_report()

## GraphX with Lin-Schatz Optimizations

In [31]:
def process_with_graph(web_rdd, num_partitions=1, max_iter=10):
    # Pregel message passing with graph frames (pyspark version of GraphX)

    # create columns for vertices and edges
    total_vertices = web_rdd.count()

    outgoing_nodes = web_rdd.groupByKey().map(lambda x: (x[0],)).collect()

    vertices = spark.createDataFrame(outgoing_nodes, ["id"])

    edges = spark.createDataFrame(web_rdd.collect(), ["src", "dst"])

    # Range partitioning
    vertices = vertices.repartitionByRange(num_partitions, "id")
    edges = edges.repartitionByRange(num_partitions, "src")

    # Build graph using graph frame
    g = gf.GraphFrame(vertices, edges)
    
    # create out degree per vertex
    out_degree = g.outDegrees
    
    # create new column out degre tracking out degree per vertex
    out_vertices = g.vertices.join(out_degree, "id", "left_outer").fillna(0.0, subset=["outDegree"])
    
    # rebuild graph with out degrees
    g = gf.GraphFrame(out_vertices, edges)
    
    # define message passing
    init_g = g.pregel.setMaxIter(max_iter).withVertexColumn("pagerank", initialExpr=lit(1.0 / total_vertices), updateAfterAggMsgsExpr=coalesce(gf.lib.Pregel.msg(), lit(0.0)))
    
    # Message passsing step
    msg_g = init_g.sendMsgToDst(gf.lib.Pregel.src("pagerank") / gf.lib.Pregel.src("outDegree"))
    
    # Aggregate across messages
    agg_g = msg_g.aggMsgs(functions.sum(gf.lib.Pregel.msg()))
    
    # execute message passing
    ranked_verts = agg_g.run()
    
    return ranked_verts

In [32]:
sc.setCheckpointDir("/tmp/graphframes-checkpoints")
stagemetrics.begin()
process_with_graph(edge_list)
stagemetrics.end()
stagemetrics.print_report()

26/04/04 17:30:26 WARN BlockManager: Block rdd_2660_0 already exists on this machine; not re-adding it
26/04/04 17:30:27 WARN CacheManager: Asked to cache already cached data.
26/04/04 17:30:29 WARN CacheManager: Asked to cache already cached data.
26/04/04 17:30:31 WARN CacheManager: Asked to cache already cached data.
26/04/04 17:30:32 WARN CacheManager: Asked to cache already cached data.
26/04/04 17:30:34 WARN CacheManager: Asked to cache already cached data.



Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 103
numTasks => 13666
elapsedTime => 11646 (12 s)
stageDuration => 12267 (12 s)
executorRunTime => 53449 (53 s)
executorCpuTime => 12548 (13 s)
executorDeserializeTime => 5615 (6 s)
executorDeserializeCpuTime => 8976 (9 s)
resultSerializationTime => 351 (0.4 s)
jvmGCTime => 2542 (3 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 2110 (2 s)
resultSize => 1007248 (983.6 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 7891974856
recordsRead => 1079
bytesRead => 586442 (572.7 KB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 2631
shuffleTotalBlocksFetched => 2469
shuffleLocalBlocksFetched => 2469
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 144626 (141.2 KB)
shuffleLocalBytesRead => 144626 (141.2 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuf

# BFS

## Naive BFS No Caching

In [18]:
def parallel_bfs(edge_rdd, source_node, max_iter):
    # TODO: Maybe pass as args so runtime measurement isnt affected
    # Create adjacency list and index
    adj_list = edge_rdd.groupByKey()

    # Initialize distances to inf
    #distances = edge_rdd.groupByKey().map(lambda x: (x[0], math.inf)).map(lambda x: (x[0], 0) if x[0] == source_node else x)
    distances = adj_list.map(lambda x: (x[0], 0 if x[0] == source_node else math.inf))

    

    for i in range(max_iter):
        
        # (node_id, ([neighbors], dist))
        node_dist = adj_list.join(distances)
        
        # dist_to_node = (node_id, dist) over neighbors
        dist_to_node = node_dist.flatMap(lambda x: [(neigh, x[1][1] + 1) for neigh in x[1][0] if x[1][1] != math.inf])

        updated_dist = distances.union(dist_to_node)
        updated_dist = updated_dist.groupByKey().mapValues(lambda x: builtins.min(x))

        # Check for convergence
        changes = updated_dist.subtract(distances)
        if changes.isEmpty():
            break

        distances = updated_dist
        
    return distances.join(adj_list)

In [19]:
stagemetrics.begin()
page_rank_results = parallel_bfs(edge_list, 0, 10)
stagemetrics.end()
stagemetrics.print_report()


Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 16
numTasks => 198
elapsedTime => 6205 (6 s)
stageDuration => 5972 (6 s)
executorRunTime => 45049 (45 s)
executorCpuTime => 913 (0.9 s)
executorDeserializeTime => 96 (96 ms)
executorDeserializeCpuTime => 120 (0.1 s)
resultSerializationTime => 58 (58 ms)
jvmGCTime => 855 (0.9 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 313 (0.3 s)
resultSize => 104174 (101.7 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 70
bytesRead => 734 (734 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 445
shuffleTotalBlocksFetched => 445
shuffleLocalBlocksFetched => 445
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 36522 (35.7 KB)
shuffleLocalBytesRead => 36522 (35.7 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesWritten => 25538 (2

## Naive BFS with Lin-Schatz

In [16]:
def ls_parallel_bfs(edge_rdd, source_node, partitions=1, max_iter=10):
    edge_rdd = edge_rdd.partitionBy(partitions, lambda key: key % partitions)
    
    # Create adjacency list and index
    adj_list = edge_rdd.groupByKey()

    # Initialize distances to inf
    distances = edge_rdd.groupByKey().map(lambda x: (x[0], math.inf)).map(lambda x: (x[0], 0) if x[0] == source_node else x)
    

    for i in range(max_iter):
        
        # (node_id, ([neighbors], dist))
        node_dist = adj_list.join(distances)
        
        # dist_to_node = (node_id, dist) over neighbors
        dist_to_node = node_dist.flatMap(lambda x: [(neigh, x[1][1] + 1) for neigh in x[1][0] if x[1][1] != math.inf])

        updated_dist = distances.union(dist_to_node)
        updated_dist = updated_dist.reduceByKey(lambda a, b: min(a, b))


        # Check for convergence
        changes = updated_dist.subtract(distances)
        if changes.isEmpty():
            break

        distances = updated_dist
        
    return distances.join(adj_list).collect()

In [17]:
stagemetrics.begin()
page_rank_results = ls_parallel_bfs(edge_list, 0, 10)
stagemetrics.end()
stagemetrics.print_report()

[Stage 3542:=================================>                   (51 + 10) / 80]


Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 24
numTasks => 1182
elapsedTime => 30170 (30 s)
stageDuration => 30073 (30 s)
executorRunTime => 278241 (4.6 min)
executorCpuTime => 3788 (4 s)
executorDeserializeTime => 661 (0.7 s)
executorDeserializeCpuTime => 823 (0.8 s)
resultSerializationTime => 404 (0.4 s)
jvmGCTime => 3702 (4 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 747 (0.7 s)
resultSize => 563404 (550.2 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 133037120
recordsRead => 70
bytesRead => 734 (734 Bytes)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 608
shuffleTotalBlocksFetched => 608
shuffleLocalBlocksFetched => 608
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 48564 (47.4 KB)
shuffleLocalBytesRead => 48564 (47.4 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesWr

## BFS with Cached Graph *

In [20]:
def parallel_bfs_caching(edge_rdd, source_node, max_iter):
    adj_list = edge_rdd.groupByKey().cache()
    distances = adj_list.map(lambda x: (x[0], math.inf)).map(lambda x: (x[0], 0) if x[0] == source_node else x).cache()

    for i in range(max_iter):
        node_dist = adj_list.join(distances)

        dist_to_node = node_dist.flatMap(lambda x: [(neigh, x[1][1] + 1) for neigh in x[1][0] if x[1][1] != math.inf])

        updated_dist = distances.union(dist_to_node).groupByKey().mapValues(lambda x: builtins.min(x)).cache()

        updated_dist.count()
        distances.unpersist()

        # Check for convergence
        changes = updated_dist.subtract(distances)
        if changes.isEmpty():
            break

        distances = updated_dist
        

    return distances.join(adj_list).collect()

In [21]:
stagemetrics.begin()
page_rank_results = parallel_bfs_caching(edge_list, 0, 10)
stagemetrics.end()
stagemetrics.print_report()



Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 21
numTasks => 280
elapsedTime => 8764 (9 s)
stageDuration => 8385 (8 s)
executorRunTime => 62986 (1.0 min)
executorCpuTime => 1138 (1 s)
executorDeserializeTime => 112 (0.1 s)
executorDeserializeCpuTime => 154 (0.2 s)
resultSerializationTime => 191 (0.2 s)
jvmGCTime => 978 (1.0 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 377 (0.4 s)
resultSize => 84953 (83.0 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 235
bytesRead => 10099 (9.9 KB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 396
shuffleTotalBlocksFetched => 396
shuffleLocalBlocksFetched => 396
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 32859 (32.1 KB)
shuffleLocalBytesRead => 32859 (32.1 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesWritten => 29077 

## Cached BFS with Lin-Schatz Optimizations

In [25]:
def ls_parallel_bfs_caching(edge_rdd, source_node, partitions=1, max_iter=10):
    edge_rdd = edge_rdd.partitionBy(partitions, lambda key: key % partitions)

    adj_list = edge_rdd.groupByKey().cache()
    distances = adj_list.map(lambda x: (x[0], math.inf)).map(lambda x: (x[0], 0) if x[0] == source_node else x).cache()

    for i in range(max_iter):
        node_dist = adj_list.join(distances)

        dist_to_node = node_dist.flatMap(lambda x: [(neigh, x[1][1] + 1) for neigh in x[1][0] if x[1][1] != math.inf])

        updated_dist = distances.union(dist_to_node).reduceByKey(lambda a, b: min(a, b)).cache()


        updated_dist.count()
        
        distances.unpersist()
        
        # Check for convergence
        changes = updated_dist.subtract(distances)
        if changes.isEmpty():
            break

        distances = updated_dist

    return distances.join(adj_list).collect()

In [26]:
stagemetrics.begin()
page_rank_results = ls_parallel_bfs_caching(edge_list, 0)
stagemetrics.end()
stagemetrics.print_report()



Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 21
numTasks => 143
elapsedTime => 6341 (6 s)
stageDuration => 5903 (6 s)
executorRunTime => 34190 (34 s)
executorCpuTime => 876 (0.9 s)
executorDeserializeTime => 104 (0.1 s)
executorDeserializeCpuTime => 96 (96 ms)
resultSerializationTime => 91 (91 ms)
jvmGCTime => 650 (0.7 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 164 (0.2 s)
resultSize => 41811 (40.8 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 199
bytesRead => 6780 (6.6 KB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 333
shuffleTotalBlocksFetched => 329
shuffleLocalBlocksFetched => 329
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 27648 (27.0 KB)
shuffleLocalBytesRead => 27648 (27.0 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesWritten => 25761 (25.2

## BFS with GraphX

In [23]:
def bfs_graphX(web_rdd, source_node, max_dist=10):

    all_nodes = web_rdd.flatMap(lambda x: [x[0], x[1]]).distinct()
    vertices = all_nodes.map(lambda x: (int(x), 0 if int(x) == source_node else float('inf'))) \
                    .toDF(["id", "distance"])
    edges = web_rdd.map(lambda x: (int(x[0]), int(x[1]), 1.0)).toDF(["src", "dst", "weights"])

    graph = gf.GraphFrame(vertices, edges)

    for i in range(max_dist):
        agg_messages = graph.aggregateMessages(
            aggCol=functions.min(gf.lib.AggregateMessages.msg).alias("min_dist"),
            sendToDst=gf.lib.AggregateMessages.src["distance"] + 1
        )
        
        new_v = graph.vertices.join(agg_messages, "id", "left") \
                      .select("id", functions.least(functions.col("distance"),
                                           functions.coalesce(functions.col("min_dist"), functions.lit(float('inf')))).alias("distance"))

        graph = gf.GraphFrame(new_v, graph.edges)

        # Checkpoint to avoid OOM error
        new_v = new_v.checkpoint()
        old_vertices = graph.vertices
        graph = gf.GraphFrame(new_v, graph.edges)
        old_vertices.unpersist()

    return graph.vertices

In [24]:
sc.setCheckpointDir("/tmp/graphframes-checkpoints")
stagemetrics.begin()
page_rank_results = bfs_graphX(edge_list, 0, 10)
stagemetrics.end()
stagemetrics.print_report()

26/04/04 22:44:09 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:10 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:11 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:12 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:13 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:13 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:14 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:15 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:16 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:16 WARN AggregateMessages: Returned DataFrame is persistent and materialized!



Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 134
numTasks => 6185
elapsedTime => 8628 (9 s)
stageDuration => 7565 (8 s)
executorRunTime => 21152 (21 s)
executorCpuTime => 5943 (6 s)
executorDeserializeTime => 10460 (10 s)
executorDeserializeCpuTime => 9824 (10 s)
resultSerializationTime => 423 (0.4 s)
jvmGCTime => 2801 (3 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 1192 (1 s)
resultSize => 1700468 (1660.6 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 1418209840
recordsRead => 1649
bytesRead => 388211 (379.1 KB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 3148
shuffleTotalBlocksFetched => 2432
shuffleLocalBlocksFetched => 2432
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 151811 (148.3 KB)
shuffleLocalBytesRead => 151811 (148.3 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffle

## GraphX BFS with Lin-Schatz Optimizations

In [20]:
def ls_bfs_graphX(web_rdd, source_node, num_partitions=1, max_dist=10):
    all_nodes = web_rdd.flatMap(lambda x: [x[0], x[1]]).distinct()
    vertices = all_nodes.map(lambda x: (int(x), 0 if int(x) == source_node else float('inf'))) \
                    .toDF(["id", "distance"])
    edges = web_rdd.map(lambda x: (int(x[0]), int(x[1]), 1.0)).toDF(["src", "dst", "weights"])

    # Range partitioning
    vertices = vertices.repartitionByRange(num_partitions, "id")
    edges = edges.repartitionByRange(num_partitions, "src")

    graph = gf.GraphFrame(vertices, edges)

    for i in range(max_dist):
        agg_messages = graph.aggregateMessages(
            aggCol=[functions.min(gf.lib.AggregateMessages.msg).alias("min_dist")],
            sendToDst=gf.lib.AggregateMessages.src["distance"] + 1
        )

        new_v = graph.vertices.join(agg_messages, "id", "left") \
                      .select("id", functions.least(functions.col("distance"),
                                           functions.coalesce(functions.col("min_dist"), functions.lit(float('inf')))).alias("distance"))

        new_v = new_v.checkpoint()
        graph = gf.GraphFrame(new_v, graph.edges)

    return graph.vertices

In [21]:
sc.setCheckpointDir("/tmp/graphframes-checkpoints")
stagemetrics.begin()
page_rank_results = ls_bfs_graphX(edge_list, 0, 10)
stagemetrics.end()
stagemetrics.print_report()

26/04/04 22:43:52 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:43:54 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:43:54 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:43:55 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:43:56 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:43:57 WARN BlockManager: Block rdd_798_1 already exists on this machine; not re-adding it
26/04/04 22:43:58 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:43:59 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:00 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:01 WARN AggregateMessages: Returned DataFrame is persistent and materialized!
26/04/04 22:44:02 WARN AggregateMessages: Returned DataFrame 


Scheduling mode = FIFO
Spark Context default degree of parallelism = 10

Aggregated Spark stage metrics:
numStages => 147
numTasks => 6832
elapsedTime => 11842 (12 s)
stageDuration => 10562 (11 s)
executorRunTime => 29037 (29 s)
executorCpuTime => 7864 (8 s)
executorDeserializeTime => 13333 (13 s)
executorDeserializeCpuTime => 11031 (11 s)
resultSerializationTime => 1133 (1 s)
jvmGCTime => 2448 (2 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 1855 (2 s)
resultSize => 1489569 (1454.7 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 1706657680
recordsRead => 2589
bytesRead => 536751 (524.2 KB)
recordsWritten => 0
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 3433
shuffleTotalBlocksFetched => 2788
shuffleLocalBlocksFetched => 2788
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 168588 (164.6 KB)
shuffleLocalBytesRead => 168588 (164.6 KB)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shu